# ChargebackOps Merchant Agent - candidate-choice reranker on Qwen2.5 3B (fp16 LoRA)

Focused training pipeline for a Google Colab T4:

1. **Phase A - JSON SFT** on heuristic rollouts. Teaches the base model the environment action schema.
2. **Phase B - candidate-choice SFT** from the saved SFT adapter. Each prompt lists valid candidate actions and the model chooses `A`, `B`, `C`, or `D`.
3. **Optional GRPO polish** is off by default. Enable it only after the candidate-choice SFT score is close to the heuristic baseline.
4. **Eval** uses the same candidate-choice interface. Do not use free-form JSON eval for the letter-choice adapter.

**Default model:** `Qwen/Qwen2.5-3B-Instruct`. If T4 memory is too tight, use `MODEL_ID=Qwen/Qwen2.5-0.5B-Instruct` only as a smoke-test fallback.

**Runtime requirement:** Google Colab T4 GPU with internet enabled. Verify with `nvidia-smi` in cell 0. If you want persistent adapters and checkpoints, mount Google Drive before running setup.


## 0. Setup - install deps + clone repo

Optional: mount Google Drive before running this cell if you want adapters and checkpoints persisted outside the Colab runtime.

In [ ]:
# GPU + repo setup for Google Colab T4.
import os
if not os.path.isdir('/content'):
    raise RuntimeError('This notebook is tuned for Google Colab. Open it in Colab or adjust WORK_DIR manually.')
WORK_DIR = '/content'
os.chdir(WORK_DIR)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_ROOT = '/content/drive/MyDrive'
PERSIST_ROOT = os.path.join(DRIVE_ROOT, 'chargebackops-artifacts') if os.path.isdir(DRIVE_ROOT) else WORK_DIR
os.makedirs(PERSIST_ROOT, exist_ok=True)

import shutil, subprocess
print('in_colab:', IN_COLAB)
print('work dir:', WORK_DIR)
print('artifact root:', PERSIST_ROOT)
print(subprocess.check_output(['nvidia-smi', '-L']).decode())

# STEP 1 - pin torch trio to the matched cu128 set. Colab pre-installs can
# drift and break the tokenizer / vision / TRL stack if left untouched.
subprocess.run(
    ['pip', 'install', '-q', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu128'],
    check=True, cwd=WORK_DIR,
)

# STEP 2a - force exact pins for the training stack. `--no-deps` keeps torch untouched.
subprocess.run(
    ['pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
     'transformers==5.5.4', 'trl==0.20.0', 'peft==0.14.0',
     'tokenizers==0.22.2', 'huggingface-hub==1.11.0'],
    check=True, cwd=WORK_DIR,
)

# STEP 2b - supporting libs with only-if-needed so torch stays put.
subprocess.run(
    ['pip', 'install', '-q', '--upgrade-strategy=only-if-needed',
     'accelerate>=0.30,<2.0', 'datasets>=2.20,<4.0',
     'matplotlib>=3.8', 'pydantic>=2.10',
     'openenv-core>=0.2.2'],
    check=True, cwd=WORK_DIR,
)

# Clone repo (always fresh).
REPO_DIR = os.path.join(WORK_DIR, 'chargebackops')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/MitudruDutta/ChargeBackOps.git', REPO_DIR],
    check=True, cwd=WORK_DIR,
)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

# Editable install with --no-deps so pyproject's app/server deps do not alter
# the training stack we just pinned.
subprocess.run(['pip', 'install', '-q', '-e', '.', '--no-deps'],
               check=True, cwd=REPO_DIR)

# Verify all critical pins land at the exact requested versions.
import importlib.metadata as md
print('torch        ', md.version('torch'))
print('torchvision  ', md.version('torchvision'))
print('transformers ', md.version('transformers'), '(want 5.5.4)')
print('tokenizers   ', md.version('tokenizers'),   '(want 0.22.2)')
print('hf-hub       ', md.version('huggingface-hub'), '(want 1.11.0)')
print('trl          ', md.version('trl'),          '(want 0.20.0)')
print('peft         ', md.version('peft'),         '(want 0.14.0)')
print('accelerate   ', md.version('accelerate'))
print('openenv-core ', md.version('openenv-core'))
assert md.version('transformers') == '5.5.4', 'transformers pin failed'
assert md.version('trl') == '0.20.0', 'trl pin failed'
assert md.version('peft') == '0.14.0', 'peft pin failed'
assert md.version('tokenizers') == '0.22.2', 'tokenizers pin failed'
assert md.version('huggingface-hub') == '1.11.0', 'huggingface-hub pin failed'


In [ ]:
# Path + module-cache flush so the editable install resolves before any other import.
import os, sys, importlib, logging, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Silence transformers per-layer "Caching is incompatible..." spam at scale.
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)
for name in [
    'transformers.models.qwen2.modeling_qwen2',
    'transformers.models.gemma4.modeling_gemma4',
]:
    logging.getLogger(name).setLevel(logging.ERROR)

REPO_DIR = globals().get('REPO_DIR', '/content/chargebackops')
PERSIST_ROOT = globals().get('PERSIST_ROOT') or (
    os.path.join('/content/drive/MyDrive', 'chargebackops-artifacts')
    if os.path.isdir('/content/drive/MyDrive') else '/content'
)
os.makedirs(PERSIST_ROOT, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Repository checkout missing at {REPO_DIR}. Run setup first.')
sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.startswith(('scenarios', 'training', 'evaluation', 'server', 'core', 'runners', 'connectors')):
        del sys.modules[mod]
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo:', REPO_DIR)
print('artifact root:', PERSIST_ROOT)


## 1. Load Qwen2.5 3B-Instruct in fp16 + attach LoRA adapter

* **fp16 base** - no `bitsandbytes`, no quantization wheel mismatch; Qwen2.5 3B is the authority-aligned default for Colab T4.
* **Chat template** - uses the selected model chat format so SFT and inference use the same prompt surface.
* **Override model** - set `MODEL_ID` in the environment before running the cell if you need a fallback.


In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-3B-Instruct')
IS_GEMMA4 = 'gemma-4' in MODEL_ID.lower()
print('MODEL_ID:', MODEL_ID)

# Gemma 4 uses AutoProcessor; most public text-only models use AutoTokenizer.
if IS_GEMMA4:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer = getattr(processor, 'tokenizer', processor)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    processor = tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # required for GRPO generation

def render_chat(messages, *, add_generation_prompt: bool) -> str:
    kwargs = {
        'tokenize': False,
        'add_generation_prompt': add_generation_prompt,
    }
    if IS_GEMMA4:
        kwargs['enable_thinking'] = False
    try:
        return processor.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking', None)
        return processor.apply_chat_template(messages, **kwargs)

def encode_text(text: str, **kwargs):
    try:
        return processor(text=text, **kwargs)
    except TypeError:
        return tokenizer(text, **kwargs)

def decode_tokens(token_ids, *, skip_special_tokens: bool = False) -> str:
    decoder = processor if hasattr(processor, 'decode') else tokenizer
    return decoder.decode(token_ids, skip_special_tokens=skip_special_tokens)

# T4 = Turing (sm_75), no bf16 hardware. fp16 only.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()  # required for LoRA + grad checkpoint

# Gemma 4 wraps projections as Gemma4ClippableLinear(linear=nn.Linear).
# PEFT 0.14 cannot LoRA-wrap the wrapper, so target the inner .linear modules.
# Public text-only defaults use normal projection names and the standard target list.
if IS_GEMMA4:
    lora_target_modules = r'.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.linear$'
    lora_rank = 8
    lora_alpha = 16
else:
    lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj']
    lora_rank = 16
    lora_alpha = 32

lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | '
      f'free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


## 2. Phase A - JSON SFT on heuristic rollouts

Builds (prompt, oracle_completion) pairs by rolling the scripted heuristic on every headline + generated task. Wraps examples in the selected model chat template so the model learns the same prompt format used at inference time.

The training cell auto-skips if `SFT_DIR/final` already exists. Set `RUN_SFT_TRAIN=1` to force retraining.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks, get_task
from training.sft_dataset import build_sft_dataset
from collections import Counter
import os

# Synthetic pool. Default is 10k rows for T4 speed; set SFT_TARGET_ROWS=32000
# or 50000 before running this cell if you have time for a larger pool.
SFT_TARGET_ROWS = int(os.environ.get('SFT_TARGET_ROWS', '10000'))
SFT_MAX_ROWS = int(os.environ.get('SFT_MAX_ROWS', str(SFT_TARGET_ROWS)))
SFT_SEED_START = int(os.environ.get('SFT_SEED_START', '1000'))
SFT_SEED_BATCH = int(os.environ.get('SFT_SEED_BATCH', '128'))
SFT_MAX_STATES_PER_TASK = int(os.environ.get('SFT_MAX_STATES_PER_TASK', '24'))
GRPO_SEED_COUNT = int(os.environ.get('GRPO_SEED_COUNT', '160'))

# Keep generated holdout/demo seeds out of training so evaluation is defensible.
HOLDOUT_SEEDS_BY_DIFF = {
    'easy': {42},
    'medium': {17, 99},
    'hard': {7, 53},
    'nightmare': {31, 77},
}
DIFFICULTIES = ['easy', 'medium', 'hard', 'nightmare']

headline_task_ids = [t.task_id for t in list_tasks()]
task_ids = list(headline_task_ids)
raw_sft = build_sft_dataset(headline_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK)
generated_train_task_ids = []

seed_cursor = SFT_SEED_START
while len(raw_sft) < SFT_TARGET_ROWS:
    batch_task_ids = []
    for diff in DIFFICULTIES:
        blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set())
        for seed in range(seed_cursor, seed_cursor + SFT_SEED_BATCH):
            if seed in blocked:
                continue
            tid = f'generated_{diff}_s{seed}'
            get_task(tid)  # fail fast if generator support breaks
            batch_task_ids.append(tid)
    raw_sft.extend(build_sft_dataset(batch_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK))
    generated_train_task_ids.extend(batch_task_ids)
    task_ids.extend(batch_task_ids)
    seed_cursor += SFT_SEED_BATCH
    print(f'generated SFT rows: {len(raw_sft):,} / target {SFT_TARGET_ROWS:,}')

if len(raw_sft) > SFT_MAX_ROWS:
    raw_sft = raw_sft[:SFT_MAX_ROWS]

# Backward-compatible seed list for GRPO curriculum generation. This is kept
# smaller than the SFT pool because GRPO rollout generation is the bottleneck.
seeds = list(range(SFT_SEED_START, SFT_SEED_START + GRPO_SEED_COUNT))

def to_chat_text(prompt: str, completion: str) -> str:
    return render_chat(
        [
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': completion},
        ],
        add_generation_prompt=False,
    )

sft_rows = [{'text': to_chat_text(s['prompt'], s['completion'])} for s in raw_sft]
sft_dataset = Dataset.from_list(sft_rows)

atype_counts = Counter(s['action_type'] for s in raw_sft)
print(f'SFT samples: {len(sft_dataset):,}, unique tasks: {len(set(s["task_id"] for s in raw_sft)):,}')
print(f'headline tasks: {len(headline_task_ids)}, generated train tasks used: {len(generated_train_task_ids):,}')
print(f'excluded generated holdout seeds: {HOLDOUT_SEEDS_BY_DIFF}')
print(f'action_type distribution: {dict(atype_counts)}')
print('sample (first 500 chars):')
print(sft_rows[0]['text'][:500])

In [ ]:
import os, torch
from trl import SFTConfig, SFTTrainer

OUT_ROOT = globals().get('PERSIST_ROOT') or (
    os.path.join('/content/drive/MyDrive', 'chargebackops-artifacts')
    if os.path.isdir('/content/drive/MyDrive') else '/content'
)
os.makedirs(OUT_ROOT, exist_ok=True)
SFT_DIR = os.path.join(OUT_ROOT, 'sft-merchant-agent')
GRPO_DIR = os.path.join(OUT_ROOT, 'grpo-merchant-agent')

# Large SFT pools should not use the old 2-epoch / 2e-4 smoke settings.
SFT_FINAL_DIR = os.path.join(SFT_DIR, 'final')
RUN_SFT_TRAIN = os.environ.get('RUN_SFT_TRAIN', 'auto').strip().lower()
TRAIN_SFT = RUN_SFT_TRAIN in {'1', 'true', 'yes', 'y', 'on'} or (
    RUN_SFT_TRAIN == 'auto' and not os.path.isdir(SFT_FINAL_DIR)
)
SFT_EPOCHS = float(os.environ.get('SFT_EPOCHS', '1'))
SFT_LR = float(os.environ.get('SFT_LR', '1e-4'))
SFT_MAX_STEPS = int(os.environ.get('SFT_MAX_STEPS', '800'))

if not TRAIN_SFT:
    print(f'Skipping SFT train; using existing adapter at {SFT_FINAL_DIR}')
else:
    sft_config = SFTConfig(
        output_dir=SFT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=SFT_EPOCHS,
        max_steps=SFT_MAX_STEPS,
        learning_rate=SFT_LR,
        logging_steps=10,
        save_steps=500,
        save_total_limit=2,
        bf16=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        max_length=1024,
        dataset_text_field='text',
        report_to='none',
        optim='adamw_torch',
        warmup_ratio=0.03,
    )
    print(f'SFT config: rows={len(sft_dataset):,}, epochs={SFT_EPOCHS}, lr={SFT_LR}, max_steps={SFT_MAX_STEPS}')

    # Silence the per-layer "Caching is incompatible with gradient checkpointing"
    # spam by disabling KV cache up-front (grad checkpoint disables it per forward
    # anyway). Same line is set before Phase B for the same reason.
    if hasattr(model, 'config'):
        model.config.use_cache = False

    sft_trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_FINAL_DIR)
    del sft_trainer
    torch.cuda.empty_cache()
    print(f'PEAK VRAM (SFT): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 2.5. Reload saved SFT, merge into base, attach fresh choice LoRA

This cell always reloads `SFT_DIR/final` into a fresh base before Phase B. That prevents accidentally continuing from, or merging, an older GRPO adapter left in memory.

The fresh LoRA starts as identity, so generation begins from the saved SFT behavior. Candidate-choice SFT then trains the new adapter on top.

In [ ]:
# Reload saved SFT LoRA into a fresh base, merge it, then attach a fresh Phase B LoRA.
from peft import PeftModel
import gc

SFT_FINAL_DIR = os.path.join(SFT_DIR, 'final')
if not os.path.isdir(SFT_FINAL_DIR):
    raise FileNotFoundError(f'Missing SFT adapter: {SFT_FINAL_DIR}. Run Phase A or upload the adapter first.')

for name in ['model', 'base_model', 'merged_base']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print(f'before SFT reload: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
sft_model = PeftModel.from_pretrained(fresh_base, SFT_FINAL_DIR)
merged_base = sft_model.merge_and_unload()
del sft_model, fresh_base
gc.collect()
torch.cuda.empty_cache()
print(f'after SFT merge: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Re-arm grad-checkpoint hook on the embedding output (lost across merge).
merged_base.enable_input_require_grads()

# Sanity: SFT-baked base should still emit clean JSON deterministically.
from training.env_adapter import build_prompt
from server.chargeback_ops_environment import ChargebackOpsEnvironment
env = ChargebackOpsEnvironment()
obs = env.reset(task_id='goods_not_received_easy')
chat = render_chat(
    [{'role': 'user', 'content': build_prompt(obs.model_dump())}],
    add_generation_prompt=True,
)
inp = encode_text(chat, return_tensors='pt').to(merged_base.device)
merged_base.eval()
with torch.no_grad():
    out = merged_base.generate(
        **inp,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print('merged-base gen:', repr(decode_tokens(out[0][inp.input_ids.shape[1]:], skip_special_tokens=False)))
merged_base.train()

# Attach fresh Phase B LoRA. lora_dropout=0 keeps candidate-choice generation deterministic.
lora_phase_b = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(merged_base, lora_phase_b)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f'after fresh Phase B LoRA: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3. Phase B - candidate-choice SFT, optional GRPO polish

Each prompt lists concrete candidate environment actions and the model chooses `A`, `B`, `C`, or `D`. The default run performs supervised choice training for 400 steps and skips GRPO (`RUN_GRPO=0`). Enable GRPO only as a short polish after the choice-SFT score is close to the heuristic baseline.

In [ ]:
from training.reward_adapter import (
    _advance_to_state,
    _heuristic_policy,
    build_state_action_dataset,
)
from training.sft_dataset import action_to_completion
from runners.baseline_runner import candidate_actions
from core.models import ChargebackOpsAction
from scenarios.simulation import list_tasks, get_task
import os
import random

# Train GRPO on the same candidate distribution used by candidate-choice eval.
GRPO_DIFFICULTIES = tuple(
    d.strip()
    for d in os.environ.get('GRPO_DIFFICULTIES', 'easy,medium,hard,nightmare').split(',')
    if d.strip()
)
curriculum_task_ids = [
    t.task_id for t in list_tasks()
    if t.difficulty in GRPO_DIFFICULTIES
]

for diff in GRPO_DIFFICULTIES:
    for s in seeds:
        blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set()) if 'HOLDOUT_SEEDS_BY_DIFF' in globals() else set()
        if s in blocked:
            continue
        tid = f'generated_{diff}_s{s}'
        try:
            get_task(tid)
            if tid not in curriculum_task_ids:
                curriculum_task_ids.append(tid)
        except Exception:
            pass

PHASE_B_MAX_STATES_PER_TASK = int(os.environ.get('PHASE_B_MAX_STATES_PER_TASK', '10'))
raw_grpo = build_state_action_dataset(curriculum_task_ids, max_states_per_task=PHASE_B_MAX_STATES_PER_TASK)

CHOICE_LABELS = ['A', 'B', 'C', 'D']
SYSTEMS = ['orders', 'payment', 'shipping', 'support', 'refunds', 'risk']
STRATEGIES = ['contest', 'accept_chargeback', 'issue_refund']

def _case_ids(obs):
    ids = [item.case_id for item in obs.queue]
    if obs.selected_case_id and obs.selected_case_id not in ids:
        ids.insert(0, obs.selected_case_id)
    return ids or ['CB-UNKNOWN']

def _evidence_ids(obs):
    visible = obs.visible_case
    if visible is None:
        return []
    ids = [item.evidence_id for item in visible.retrieved_evidence]
    ids += [item.evidence_id for item in visible.attached_evidence]
    return list(dict.fromkeys(ids))

def _candidate_for(action_type, obs, offset):
    case_ids = _case_ids(obs)
    case_id = obs.selected_case_id or case_ids[offset % len(case_ids)]
    evidence_ids = _evidence_ids(obs)
    if action_type == 'select_case':
        return ChargebackOpsAction(action_type='select_case', case_id=case_ids[offset % len(case_ids)])
    if action_type == 'wait_for_updates':
        return ChargebackOpsAction(action_type='wait_for_updates')
    if action_type == 'query_system':
        return ChargebackOpsAction(
            action_type='query_system',
            case_id=case_id,
            system_name=SYSTEMS[offset % len(SYSTEMS)],
        )
    if action_type in ('set_strategy', 'resolve_case'):
        return ChargebackOpsAction(
            action_type=action_type,
            case_id=case_id,
            strategy=STRATEGIES[offset % len(STRATEGIES)],
        )
    if action_type in ('add_evidence', 'remove_evidence'):
        return ChargebackOpsAction(
            action_type=action_type,
            case_id=case_id,
            evidence_ids=evidence_ids[:2] or ['NO-EVIDENCE'],
        )
    if action_type == 'respond_to_pre_arb':
        return ChargebackOpsAction(
            action_type='respond_to_pre_arb',
            case_id=case_id,
            compelling_evidence_ids=evidence_ids[:2] or ['NO-EVIDENCE'],
            note='Respond with the strongest available compelling evidence.',
        )
    return ChargebackOpsAction(action_type=action_type, case_id=case_id)

def _unique_action_json(actions):
    seen = set()
    out = []
    for action in actions:
        payload = action_to_completion(action)
        if payload not in seen:
            seen.add(payload)
            out.append(payload)
    return out

def _candidate_bundle(task_id, state_step):
    advanced = _advance_to_state(task_id, int(state_step))
    if advanced is None:
        return None
    _env, obs = advanced
    ranked = candidate_actions(obs.model_dump())
    if len(ranked) < 2:
        return None
    oracle_json = action_to_completion(ranked[0].action)
    candidate_pairs = []
    seen = set()
    for candidate in ranked[:6]:
        payload = action_to_completion(candidate.action)
        if payload in seen:
            continue
        seen.add(payload)
        candidate_pairs.append((payload, candidate.summary))
        if len(candidate_pairs) >= 4:
            break
    if len(candidate_pairs) < 2 or oracle_json not in {payload for payload, _ in candidate_pairs}:
        return None
    rng = random.Random(f'{task_id}:{state_step}')
    rng.shuffle(candidate_pairs)
    candidate_jsons = [payload for payload, _summary in candidate_pairs]
    candidate_summaries = [summary for _payload, summary in candidate_pairs]
    target_idx = candidate_jsons.index(oracle_json)
    return {
        'candidate_jsons': candidate_jsons,
        'candidate_summaries': candidate_summaries,
        'target_label': CHOICE_LABELS[target_idx],
    }

def to_choice_prompt(prompt: str, candidate_jsons: list[str], candidate_summaries: list[str] | None = None) -> str:
    if candidate_summaries is None:
        candidate_summaries = [''] * len(candidate_jsons)
    candidate_lines = [
        f'{label}: {summary}\n{candidate}' if summary else f'{label}: {candidate}'
        for label, candidate, summary in zip(CHOICE_LABELS, candidate_jsons, candidate_summaries)
    ]
    content = (
        'You are a ChargeBackOps policy reranker. Choose the best next environment action.\n'
        + 'Use the observation and candidate action details. Return only one letter.\n\n'
        + prompt
        + '\n\nCANDIDATE MODE: Do not write JSON. Choose the single best candidate action below.\n'
        + '\n'.join(candidate_lines)
        + '\nReturn only one letter: '
        + ', '.join(CHOICE_LABELS[:len(candidate_jsons)])
        + '.\nANSWER:'
    )
    return render_chat(
        [{'role': 'user', 'content': content}],
        add_generation_prompt=True,
    )

grpo_rows = []
for sample in raw_grpo:
    bundle = _candidate_bundle(sample['task_id'], int(sample['state_step']))
    if bundle is None:
        continue
    prompt = to_choice_prompt(sample['prompt'], bundle['candidate_jsons'], bundle['candidate_summaries'])
    n_tokens = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    if n_tokens <= 1000:
        grpo_rows.append(
            {
                'prompt': prompt,
                'task_id': sample['task_id'],
                'state_step': int(sample['state_step']),
                'candidate_jsons': bundle['candidate_jsons'],
                'candidate_summaries': bundle['candidate_summaries'],
                'target_label': bundle['target_label'],
            }
        )

grpo_dataset = Dataset.from_list(grpo_rows)
unique_tasks = len({row['task_id'] for row in grpo_rows})
print(f'GRPO candidate-choice samples: {len(grpo_dataset)}, unique tasks: {unique_tasks}, difficulties={GRPO_DIFFICULTIES}, max_states_per_task={PHASE_B_MAX_STATES_PER_TASK}')
print('sample prompt tail:', repr(grpo_rows[0]['prompt'][-120:]) if grpo_rows else 'EMPTY')

In [ ]:
import sys, types, importlib.machinery, importlib.metadata as md, re
from contextlib import contextmanager, nullcontext
from training.reward_adapter import compute_reward
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import LogitsProcessor, LogitsProcessorList

# Colab can leave stale optional-package metadata around. Plain GRPO on T4
# does not need vLLM, mergekit, llm_blender, deepspeed, or liger kernels.
OPTIONAL_PREFIXES = (
    'trl', 'vllm', 'mergekit', 'llm_blender', 'weave',
    'liger_kernel', 'deepspeed', 'math_verify', 'vllm_ascend',
)
for mod in list(sys.modules):
    if mod == 'trl' or mod.startswith(OPTIONAL_PREFIXES):
        del sys.modules[mod]

import trl.import_utils as trl_import_utils
for attr in [
    '_vllm_available', '_vllm_ascend_available', '_mergekit_available',
    '_llm_blender_available', '_weave_available', '_liger_kernel_available',
    '_deepspeed_available', '_math_verify_available',
]:
    if hasattr(trl_import_utils, attr):
        setattr(trl_import_utils, attr, False)
trl_import_utils._vllm_version = '0.0.0'
trl_import_utils.is_vllm_available = lambda: False
trl_import_utils.is_vllm_ascend_available = lambda: False
trl_import_utils.is_mergekit_available = lambda: False
trl_import_utils.is_llm_blender_available = lambda: False
trl_import_utils.is_weave_available = lambda: False
trl_import_utils.is_liger_kernel_available = lambda *args, **kwargs: False
trl_import_utils.is_deepspeed_available = lambda: False
trl_import_utils.is_math_verify_available = lambda: False

def _stub_module(name):
    module = types.ModuleType(name)
    module.__spec__ = importlib.machinery.ModuleSpec(name, loader=None)
    return module

for name in [
    'vllm', 'vllm.distributed', 'vllm.distributed.device_communicators',
    'vllm.distributed.device_communicators.pynccl', 'vllm.distributed.utils',
    'mergekit', 'mergekit.config', 'mergekit.merge', 'llm_blender',
]:
    sys.modules.setdefault(name, _stub_module(name))

class _DisabledOptional:
    def __init__(self, *args, **kwargs):
        raise RuntimeError('Optional dependency disabled for this Colab T4 run.')

sys.modules['vllm.distributed.device_communicators.pynccl'].PyNcclCommunicator = _DisabledOptional
sys.modules['vllm.distributed.utils'].StatelessProcessGroup = _DisabledOptional
sys.modules['mergekit.config'].MergeConfiguration = _DisabledOptional
sys.modules['mergekit.merge'].MergeOptions = _DisabledOptional
sys.modules['mergekit.merge'].run_merge = lambda *args, **kwargs: (_ for _ in ()).throw(
    RuntimeError('mergekit disabled for this Colab T4 run.')
)

from trl.trainer.grpo_config import GRPOConfig
from trl.trainer.grpo_trainer import GRPOTrainer
from trl import SFTConfig, SFTTrainer
import trl.trainer.grpo_trainer as grpo_trainer_module
from trl.data_utils import maybe_apply_chat_template
from trl.extras.profiling import profiling_context
from trl.models import unwrap_model_for_generation
try:
    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
except Exception:
    FSDP = None
print('trl:', md.version('trl'), '| GRPO import OK')

def _choice_index(completion: str, candidate_count: int):
    # The constrained trainer emits one choice token plus EOS. This parser is
    # intentionally tolerant so older smoke outputs can still be diagnosed.
    text = (completion or '').strip().upper()
    if not text:
        return None
    match = re.search(r'\b([ABCD])\b', text)
    if match is None and text[0] in CHOICE_LABELS:
        label = text[0]
    elif match is not None:
        label = match.group(1)
    else:
        return None
    idx = CHOICE_LABELS.index(label)
    return idx if idx < candidate_count else None

def choice_task_reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    state_steps = kwargs.get('state_step') or kwargs.get('state_steps')
    candidate_jsons = kwargs.get('candidate_jsons')
    target_labels = kwargs.get('target_label') or kwargs.get('target_labels')
    rewards = []
    iterable = zip(prompts, completions, task_ids, state_steps, candidate_jsons, target_labels or [None] * len(prompts))
    for prompt, completion, task_id, state_step, candidates, target_label in iterable:
        idx = _choice_index(completion, len(candidates))
        if idx is None:
            rewards.append(0.0)
            continue
        if target_label in CHOICE_LABELS:
            rewards.append(1.0 if CHOICE_LABELS[idx] == target_label else 0.0)
            continue
        reward = compute_reward(
            [prompt],
            [candidates[idx]],
            task_ids=[task_id],
            state_steps=[state_step],
        )[0]
        rewards.append(reward)
    return rewards

def choice_format_reward_fn(prompts, completions, **kwargs):
    candidate_jsons = kwargs.get('candidate_jsons')
    rewards = []
    for completion, candidates in zip(completions, candidate_jsons):
        raw = completion or ''
        eos_text = getattr(tokenizer, 'eos_token', None) or ''
        stripped = raw.replace(eos_text, '').strip().upper() if eos_text else raw.strip().upper()
        idx = _choice_index(raw, len(candidates))
        reward = 0.05 if idx is not None else -0.10
        if stripped in CHOICE_LABELS[:len(candidates)]:
            reward += 0.15
        rewards.append(reward)
    return rewards

def _as_int_token(token_id):
    if isinstance(token_id, (list, tuple)):
        return int(token_id[0])
    return int(token_id)

class ChoiceOnlyLogitsProcessor(LogitsProcessor):
    # Permanent fix for Qwen multilingual/junk spillover: invalid tokens are
    # masked before sampling, not punished after sampling.
    def __init__(self, choice_token_ids, eos_token_id, prompt_length):
        self.choice_token_ids = [int(token_id) for token_id in choice_token_ids]
        self.eos_token_id = _as_int_token(eos_token_id)
        self.prompt_length = int(prompt_length)

    def __call__(self, input_ids, scores):
        generated = input_ids.shape[1] - self.prompt_length
        allowed = self.choice_token_ids if generated == 0 else [self.eos_token_id]
        masked = scores.new_full(scores.shape, float('-inf'))
        masked[:, allowed] = scores[:, allowed]
        return masked

class ChoiceConstrainedGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, choice_token_ids, **kwargs):
        super().__init__(*args, **kwargs)
        self.choice_token_ids = [int(token_id) for token_id in choice_token_ids]

    def _generate_single_turn(self, prompts, images):
        if images is not None:
            raise ValueError('ChoiceConstrainedGRPOTrainer only supports text prompts.')
        device = self.accelerator.device
        prompts_text = [
            maybe_apply_chat_template({'prompt': prompt}, self.processing_class)['prompt']
            for prompt in prompts
        ]
        generate_inputs = self.processing_class(
            text=prompts_text,
            return_tensors='pt',
            padding=True,
            padding_side='left',
            max_length=self.max_prompt_length,
            truncation=True,
            add_special_tokens=False,
        )
        generate_inputs = super()._prepare_inputs(generate_inputs)
        prompt_ids, prompt_mask = generate_inputs['input_ids'], generate_inputs['attention_mask']
        prompt_length = prompt_ids.size(1)
        logits_processor = LogitsProcessorList([
            ChoiceOnlyLogitsProcessor(self.choice_token_ids, self.eos_token_id, prompt_length)
        ])
        fsdp_context = (
            FSDP.summon_full_params(self.model_wrapped, recurse=False)
            if FSDP is not None and self.is_fsdp_enabled else nullcontext()
        )
        with (
            profiling_context(self, 'choice_constrained_generate'),
            unwrap_model_for_generation(
                self.model_wrapped,
                self.accelerator,
                gather_deepspeed3_params=self.args.ds3_gather_for_generation,
            ) as unwrapped_model,
            torch.no_grad(),
            fsdp_context,
        ):
            prompt_completion_ids = unwrapped_model.generate(
                **generate_inputs,
                generation_config=self.generation_config,
                logits_processor=logits_processor,
                disable_compile=True,
            )
        completion_ids = prompt_completion_ids[:, prompt_length:]
        eos_token_id = _as_int_token(self.eos_token_id)
        is_eos = completion_ids == eos_token_id
        eos_idx = torch.full((is_eos.size(0),), is_eos.size(1), dtype=torch.long, device=device)
        eos_idx[is_eos.any(dim=1)] = is_eos.int().argmax(dim=1)[is_eos.any(dim=1)]
        sequence_indices = torch.arange(is_eos.size(1), device=device).expand(is_eos.size(0), -1)
        completion_mask = (sequence_indices <= eos_idx.unsqueeze(1)).int()
        prompt_ids = [p[m].tolist() for p, m in zip(prompt_ids, prompt_mask.bool())]
        completion_ids = [c[m].tolist() for c, m in zip(completion_ids, completion_mask.bool())]
        return prompt_ids, completion_ids, None, {}

choice_token_ids = []
for label in CHOICE_LABELS:
    ids = tokenizer(label, add_special_tokens=False)['input_ids']
    if len(ids) != 1:
        raise ValueError(f'Choice label {label!r} is not one token: {ids}')
    choice_token_ids.append(ids[0])
print('choice token ids:', dict(zip(CHOICE_LABELS, choice_token_ids)))

def _make_choice_constrained_generate(original_generate):
    def constrained_generate(*args, **kwargs):
        input_ids = kwargs.get('input_ids')
        if input_ids is None and args and torch.is_tensor(args[0]):
            input_ids = args[0]
        if input_ids is not None:
            existing = kwargs.get('logits_processor')
            processors = list(existing) if existing is not None else []
            processors.append(
                ChoiceOnlyLogitsProcessor(choice_token_ids, tokenizer.eos_token_id, input_ids.shape[1])
            )
            kwargs['logits_processor'] = LogitsProcessorList(processors)
        return original_generate(*args, **kwargs)
    return constrained_generate

_original_unwrap_model_for_generation = grpo_trainer_module.unwrap_model_for_generation

@contextmanager
def _choice_constrained_unwrap_model_for_generation(*args, **kwargs):
    with _original_unwrap_model_for_generation(*args, **kwargs) as unwrapped_model:
        original_generate = unwrapped_model.generate
        unwrapped_model.generate = _make_choice_constrained_generate(original_generate)
        try:
            yield unwrapped_model
        finally:
            unwrapped_model.generate = original_generate

# TRL 0.20 performs generation inside _generate_and_score_completions rather
# than _generate_single_turn. Replace the module-level unwrap hook so the same
# hard token constraint is applied on that older code path too.
grpo_trainer_module.unwrap_model_for_generation = _choice_constrained_unwrap_model_for_generation

# Re-arm hooks and disable cache/noisy stochastic modules.
model.enable_input_require_grads()
if hasattr(model, 'config'):
    model.config.use_cache = False
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}
if hasattr(model, 'base_model') and not hasattr(model.base_model, 'warnings_issued'):
    model.base_model.warnings_issued = {}
if hasattr(model, 'get_base_model'):
    base = model.get_base_model()
    if not hasattr(base, 'warnings_issued'):
        base.warnings_issued = {}

for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.p = 0.0

# Short supervised warmup on the exact A/B/C/D candidate task. This is the
# missing bridge between JSON SFT and GRPO: RL should refine a policy that
# already understands shuffled candidates, not discover the interface from scratch.
RUN_CHOICE_SFT_WARMUP = os.environ.get('RUN_CHOICE_SFT_WARMUP', '1').strip().lower() not in {'0', 'false', 'no'}
CHOICE_SFT_STEPS = int(os.environ.get('CHOICE_SFT_STEPS', '400'))
CHOICE_SFT_LR = float(os.environ.get('CHOICE_SFT_LR', '1e-5'))
CHOICE_SFT_SCHEDULER = os.environ.get('CHOICE_SFT_SCHEDULER', 'constant')
if RUN_CHOICE_SFT_WARMUP and CHOICE_SFT_STEPS > 0 and grpo_rows:
    choice_sft_rows = [
        {'text': row['prompt'] + row['target_label'] + tokenizer.eos_token}
        for row in grpo_rows
        if row.get('target_label') in CHOICE_LABELS
    ]
    choice_sft_dataset = Dataset.from_list(choice_sft_rows)
    choice_sft_config = SFTConfig(
        output_dir=os.path.join(GRPO_DIR, 'choice-sft-warmup'),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        max_steps=CHOICE_SFT_STEPS,
        learning_rate=CHOICE_SFT_LR,
        logging_steps=10,
        save_steps=CHOICE_SFT_STEPS,
        save_total_limit=1,
        bf16=False,
        fp16=True,
        max_grad_norm=0.5,
        gradient_checkpointing=False,
        max_length=1024,
        dataset_text_field='text',
        report_to='none',
        optim='adamw_torch',
        warmup_ratio=0.03,
        lr_scheduler_type=CHOICE_SFT_SCHEDULER,
    )
    print(f'choice-SFT warmup: rows={len(choice_sft_dataset):,}, steps={CHOICE_SFT_STEPS}, lr={CHOICE_SFT_LR}, scheduler={CHOICE_SFT_SCHEDULER}')
    choice_sft_trainer = SFTTrainer(
        model=model,
        args=choice_sft_config,
        train_dataset=choice_sft_dataset,
        processing_class=tokenizer,
    )
    choice_sft_trainer.train()
    choice_sft_trainer.save_model(os.path.join(GRPO_DIR, 'choice-sft-final'))
    del choice_sft_trainer
    torch.cuda.empty_cache()
    model.enable_input_require_grads()
    if hasattr(model, 'config'):
        model.config.use_cache = False

# Verify the PEFT-wrapped model is sane before TRL takes over generation.
if grpo_rows:
    model.eval()
    sanity_inputs = encode_text(grpo_rows[0]['prompt'], return_tensors='pt').to(model.device)
    sanity_logits_processor = LogitsProcessorList([
        ChoiceOnlyLogitsProcessor(choice_token_ids, tokenizer.eos_token_id, sanity_inputs.input_ids.shape[1])
    ])
    with torch.no_grad():
        sanity_out = model.generate(
            **sanity_inputs,
            max_new_tokens=2,
            do_sample=True,
            temperature=1.0,
            logits_processor=sanity_logits_processor,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    sanity_completion = decode_tokens(
        sanity_out[0][sanity_inputs.input_ids.shape[1]:],
        skip_special_tokens=False,
    )
    print('pre-GRPO choice sanity:', repr(sanity_completion[:80]))
    print('candidate_jsons:', grpo_rows[0]['candidate_jsons'])
    model.train()

grpo_config = GRPOConfig(
    output_dir=GRPO_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=8,
    max_prompt_length=1024,
    max_completion_length=2,
    learning_rate=float(os.environ.get('GRPO_LR', '1e-7')),
    max_steps=int(os.environ.get('GRPO_MAX_STEPS', '60')),
    logging_steps=5,
    save_steps=40,
    save_total_limit=3,
    bf16=False,
    fp16=True,
    max_grad_norm=0.5,
    gradient_checkpointing=False,
    report_to='none',
    beta=0.0,
    temperature=2.0,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    use_vllm=False,
    generation_kwargs={
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
    },
    log_completions=True,
    num_completions_to_print=2,
    optim='adamw_torch',
    lr_scheduler_type='constant',
)

RUN_GRPO = os.environ.get('RUN_GRPO', '0').strip().lower() not in {'0', 'false', 'no'}
if RUN_GRPO:
    grpo_trainer = ChoiceConstrainedGRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[choice_format_reward_fn, choice_task_reward_fn],
        args=grpo_config,
        train_dataset=grpo_dataset,
        choice_token_ids=choice_token_ids,
    )
    grpo_trainer.train()
else:
    print('RUN_GRPO=0: skipped GRPO after choice-SFT warmup')
print(f'PEAK VRAM (GRPO): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 3.5. Candidate-choice eval

Evaluate the in-memory candidate-choice model with the same `A/B/C/D` interface used during training. This is the score to trust for the Phase B adapter.

In [ ]:
from statistics import mean
from collections import defaultdict
import random
from runners.baseline_runner import candidate_actions
from training.env_adapter import build_prompt
from training.sft_dataset import action_to_completion
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from scenarios.simulation import list_tasks

def _candidate_choice_action(obs, *, task_id: str, step: int):
    candidates = candidate_actions(obs.model_dump())[:4]
    if not candidates:
        return None, None
    rng = random.Random(f'eval:{task_id}:{step}')
    rng.shuffle(candidates)
    if len(candidates) == 1:
        return candidates[0].action, 'single'
    candidate_jsons = [action_to_completion(candidate.action) for candidate in candidates]
    candidate_summaries = [candidate.summary for candidate in candidates]
    choice_prompt = to_choice_prompt(build_prompt(obs.model_dump()), candidate_jsons, candidate_summaries)
    inputs = encode_text(choice_prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    logits_processor = LogitsProcessorList([
        ChoiceOnlyLogitsProcessor(choice_token_ids, tokenizer.eos_token_id, inputs.input_ids.shape[1])
    ])
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            logits_processor=logits_processor,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    completion = decode_tokens(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    idx = _choice_index(completion, len(candidates))
    if idx is None:
        return candidates[0].action, f'invalid:{completion!r}'
    return candidates[idx].action, completion

def evaluate_candidate_choice_model(task_ids=None):
    model.eval()
    tasks = list_tasks() if task_ids is None else [t for t in list_tasks() if t.task_id in set(task_ids)]
    rows = []
    for task in tasks:
        env = ChargebackOpsEnvironment()
        obs = env.reset(task_id=task.task_id)
        steps = 0
        invalid = 0
        while not obs.done and steps < task.max_steps + 5:
            action, raw_choice = _candidate_choice_action(obs, task_id=task.task_id, step=steps)
            if action is None:
                invalid += 1
                break
            if isinstance(raw_choice, str) and raw_choice.startswith('invalid:'):
                invalid += 1
            obs = env.step(action)
            steps += 1
        score = obs.grader_report.normalized_score if obs.grader_report else 0.0
        rows.append({'task_id': task.task_id, 'difficulty': task.difficulty, 'score': float(score), 'steps': steps, 'invalid': invalid})
    return rows

candidate_eval_rows = evaluate_candidate_choice_model()
print('CANDIDATE-CHOICE EVAL')
print(f"overall mean={mean(row['score'] for row in candidate_eval_rows):.4f}")
by_family = defaultdict(list)
for row in candidate_eval_rows:
    by_family[row['difficulty']].append(row['score'])
for family in sorted(by_family):
    print(f"{family}: mean={mean(by_family[family]):.4f} n={len(by_family[family])}")
for row in candidate_eval_rows:
    print(f"  {row['task_id']}: score={row['score']:.3f} steps={row['steps']} invalid={row['invalid']}")


## 4. Optional free-form per-checkpoint eval - overall + per-family

This is the legacy JSON-action evaluator for base/SFT checkpoints. Candidate-choice Phase B adapters emit only `A`/`B`/`C`/`D`, so score them with the candidate-choice eval cell above. This cell is skipped by default; set `RUN_FREEFORM_CHECKPOINT_EVAL=1` to run it.

In [ ]:
import glob, re
from peft import PeftModel
from transformers import AutoProcessor, AutoTokenizer
from training.curve import (
    evaluate_checkpoint, evaluate_checkpoint_by_family,
    plot_training_curve, plot_training_curve_by_family,
)

TRUE_FLAGS = {'1', 'true', 'yes', 'y', 'on'}
RUN_FREEFORM_CHECKPOINT_EVAL = os.environ.get('RUN_FREEFORM_CHECKPOINT_EVAL', '0').strip().lower() in TRUE_FLAGS
INCLUDE_GRPO_IN_FREEFORM_EVAL = os.environ.get('INCLUDE_GRPO_IN_FREEFORM_EVAL', '0').strip().lower() in TRUE_FLAGS

# Legacy JSON-action evaluator. Candidate-choice Phase B emits letters, not JSON.
def make_text_policy(adapter_path: str | None, adapter_kind: str = 'base'):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map='auto', trust_remote_code=True,
    )
    if adapter_kind == 'grpo':
        sft_model = PeftModel.from_pretrained(base, os.path.join(SFT_DIR, 'final'))
        base = sft_model.merge_and_unload()
        m = PeftModel.from_pretrained(base, adapter_path)
    elif adapter_path is not None:
        m = PeftModel.from_pretrained(base, adapter_path)
    else:
        m = base
    m.eval()

    if IS_GEMMA4:
        proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        tok = getattr(proc, 'tokenizer', proc)
    else:
        tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        proc = tok
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    def local_render(messages, *, add_generation_prompt: bool) -> str:
        kwargs = {'tokenize': False, 'add_generation_prompt': add_generation_prompt}
        if IS_GEMMA4:
            kwargs['enable_thinking'] = False
        try:
            return proc.apply_chat_template(messages, **kwargs)
        except TypeError:
            kwargs.pop('enable_thinking', None)
            return proc.apply_chat_template(messages, **kwargs)

    def local_encode(text: str, **kwargs):
        try:
            return proc(text=text, **kwargs)
        except TypeError:
            return tok(text, **kwargs)

    def local_decode(token_ids, *, skip_special_tokens: bool = True) -> str:
        decoder = proc if hasattr(proc, 'decode') else tok
        return decoder.decode(token_ids, skip_special_tokens=skip_special_tokens)

    def policy(prompt: str) -> str:
        chat = local_render(
            [{'role': 'user', 'content': prompt}],
            add_generation_prompt=True,
        )
        inputs = local_encode(chat, return_tensors='pt', truncation=True, max_length=1024).to(m.device)
        with torch.no_grad():
            out = m.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id,
            )
        return local_decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return policy, m

# Catalog of checkpoints: untrained base and SFT final by default.
ckpt_specs = [('base', None, 0, 'base')]
ckpt_specs.append(('sft', os.path.join(SFT_DIR, 'final'), 1, 'sft'))
grpo_dirs = sorted(
    glob.glob(os.path.join(GRPO_DIR, 'checkpoint-*')),
    key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)),
)
if INCLUDE_GRPO_IN_FREEFORM_EVAL:
    for d in grpo_dirs:
        step = int(re.search(r'checkpoint-(\d+)', d).group(1))
        ckpt_specs.append((f'grpo-{step}', d, 1 + step, 'grpo'))
elif grpo_dirs:
    print(f'Found {len(grpo_dirs)} GRPO checkpoint(s); not adding them to free-form eval.')

if not RUN_FREEFORM_CHECKPOINT_EVAL:
    print('Skipping free-form checkpoint eval.')
    print('Use the CANDIDATE-CHOICE EVAL cell above for letter-choice Phase B adapters.')
    print('Set RUN_FREEFORM_CHECKPOINT_EVAL=1 to evaluate base/SFT JSON policies.')
else:
    overall = []
    grouped = []
    for label, path, step, kind in ckpt_specs:
        print(f'eval {label} from {path}')
        pol, m_ckpt = make_text_policy(path, kind)
        overall.append(evaluate_checkpoint(step=step, policy=pol))
        grouped.append(evaluate_checkpoint_by_family(step=step, policy=pol))
        del m_ckpt
        torch.cuda.empty_cache()

    print('\nOVERALL CURVE:')
    for c in overall:
        print(f'  step={c.step:4d} mean={c.mean_score:.4f}')

    print('\nPER-FAMILY CURVE:')
    for g in grouped:
        line = f'  step={g.step:4d}'
        for fam in sorted(g.by_family.keys()):
            line += f'  {fam}={g.by_family[fam].mean_score:.3f}'
        print(line)

    from runners.benchmark_runner import run_policy_sweep
    sweep = run_policy_sweep()
    heur_overall = next(s.mean_score for s in sweep.policies if s.policy == 'heuristic')

    FIG_DIR = os.path.join(REPO_DIR, 'docs', 'figures')
    os.makedirs(FIG_DIR, exist_ok=True)
    plot_training_curve(
        overall, os.path.join(FIG_DIR, 'training_curve.png'),
        baseline_scores={'heuristic': heur_overall, 'naive': 0.0},
    )
    plot_training_curve_by_family(
        grouped, os.path.join(FIG_DIR, 'training_curve_by_family.png'),
        family_order=['easy', 'medium', 'hard', 'nightmare'],
    )
    print(f'\nfigures saved to {FIG_DIR}/')


## 5. Optional free-form checkpoint diagnostic

For JSON-action policies only. Candidate-choice Phase B adapters should be diagnosed with candidate prompts, not this free-form JSON completion check. This cell is skipped by default; set `RUN_FREEFORM_DIAGNOSTIC=1` to run it.

In [ ]:
TRUE_FLAGS = globals().get('TRUE_FLAGS', {'1', 'true', 'yes', 'y', 'on'})
RUN_FREEFORM_DIAGNOSTIC = os.environ.get('RUN_FREEFORM_DIAGNOSTIC', '0').strip().lower() in TRUE_FLAGS

if not RUN_FREEFORM_DIAGNOSTIC:
    print('Skipping free-form checkpoint diagnostic.')
    print('Use candidate-choice prompts for letter-choice GRPO adapters.')
else:
    from training.env_adapter import build_prompt, parse_completion
    from training.reward_adapter import compute_reward
    from server.chargeback_ops_environment import ChargebackOpsEnvironment
    from runners.benchmark_runner import heuristic_policy

    final_adapter = os.path.join(SFT_DIR, 'final')
    final_kind = 'sft'
    if INCLUDE_GRPO_IN_FREEFORM_EVAL and grpo_dirs:
        final_adapter = grpo_dirs[-1]
        final_kind = 'grpo'

    print(f'diagnose adapter: {final_adapter}')
    policy, m = make_text_policy(final_adapter, final_kind)

    for tid in ['goods_not_received_easy', 'queue_optimization_hard', 'generated_nightmare_s31']:
        env = ChargebackOpsEnvironment()
        obs = env.reset(task_id=tid)
        raw = build_prompt(obs.model_dump())
        completion = policy(raw)
        parsed = parse_completion(completion)
        oracle = heuristic_policy(obs.model_dump())
        reward = compute_reward(['x'], [completion], task_ids=[tid], state_steps=[0])[0]
        print(f'\n=== {tid} ===')
        print(f'oracle: {oracle.action_type} case={oracle.case_id}')
        print(f'completion (first 200): {repr(completion[:200])}')
        print(f'parsed: {parsed}')
        print(f'reward vs oracle: {reward:.3f}')
    del m
    torch.cuda.empty_cache()

## Done

Artifacts are written under `PERSIST_ROOT`:
* default runtime path: `/content/`
* Drive-backed path when mounted before setup: `/content/drive/MyDrive/chargebackops-artifacts/`

Default artifacts:
* `PERSIST_ROOT/sft-merchant-agent/final/` - Phase A JSON SFT adapter.
* `PERSIST_ROOT/grpo-merchant-agent/choice-sft-final/` - Phase B candidate-choice adapter.

Optional artifacts:
* `PERSIST_ROOT/grpo-merchant-agent/checkpoint-*` - created only when `RUN_GRPO=1`.
* `/content/chargebackops/docs/figures/training_curve*.png` - created only when `RUN_FREEFORM_CHECKPOINT_EVAL=1`.

Trust the `## 3.5. Candidate-choice eval` score for the Phase B adapter. Do not use free-form JSON diagnostics as the final score for this letter-choice model.
